# Generate spring mass matricies

In [1]:
import numpy as np

def stationary_covariance_ou(A, Sigma):
    """Solve A P + P A^T = Sigma Sigma^T for the OU stationary covariance."""
    state_dim = A.shape[0]
    rhs = (Sigma @ Sigma.T).reshape(-1)
    system = np.kron(np.eye(state_dim), A) + np.kron(A, np.eye(state_dim))
    cov = np.linalg.solve(system, rhs).reshape(state_dim, state_dim)
    return 0.5 * (cov + cov.T)

def generate_spring_mass(
    num_masses,
    seed=None,
    rng=None,
    mass_range=(0.8, 1.2),
    spring_range=(0.8, 1.2),
    damping_range=(0.15, 0.25)
):
    """Generate the linear spring-mass chain as an OU-style state-space model.

    State ordering is x = [q_1, ..., q_r, v_1, ..., v_r], where r = num_masses.
    The simulator uses drift b(x) = A (mu - x), so the physical linear dynamics are
    recovered by choosing A = -[[0, I], [A21, A22]].
    """
    if num_masses < 1:
        raise ValueError("num_masses must be at least 1")

    if rng is None:
        rng = np.random.default_rng(seed)

    masses = rng.uniform(*mass_range, size=num_masses)
    springs = rng.uniform(*spring_range, size=num_masses + 1)
    damping = rng.uniform(*damping_range, size=num_masses + 1)

    A21 = np.zeros((num_masses, num_masses))
    A22 = np.zeros((num_masses, num_masses))

    for i in range(num_masses):
        A21[i, i] = -(springs[i] + springs[i + 1]) / masses[i]
        A22[i, i] = -(damping[i] + damping[i + 1]) / masses[i]
        if i < num_masses - 1:
            A21[i, i + 1] = springs[i + 1] / masses[i]
            A21[i + 1, i] = springs[i + 1] / masses[i + 1]

    A = -np.block([
        [np.zeros((num_masses, num_masses)), np.eye(num_masses)],
        [A21, A22],
    ])

    state_dim = 2 * num_masses
    mu = np.zeros(state_dim)
    Sigma = np.eye(state_dim)

    config = {
        "name": "ou",
        "A": A.tolist(),
        "mu": mu.tolist(),
        "Sigma": Sigma.tolist(),
    }

    cov_ss = stationary_covariance_ou(A, Sigma)

    return config, cov_ss.tolist()


In [2]:
spring_1_masses, cov_ss = generate_spring_mass(num_masses=1, seed=20260414)
print(spring_1_masses)
print(cov_ss)

{'name': 'ou', 'A': [[-0.0, -1.0], [2.378002222780105, 0.515901172837531]], 'mu': [0.0, 0.0], 'Sigma': [[1.0, 0.0], [0.0, 1.0]]}
[[1.4852112217918285, -0.4999999999999999], [-0.4999999999999999, 3.2738850003001585]]


In [3]:
spring_2_masses, cov_ss = generate_spring_mass(num_masses=2, seed=20260414)
print(spring_2_masses)
print(cov_ss)

{'name': 'ou', 'A': [[-0.0, -0.0, -1.0, -0.0], [-0.0, -0.0, -0.0, -1.0], [2.2641560053474103, -1.187871250391528, 0.488235120883477, -0.0], [-0.9125415305189191, 1.941231842218933, -0.0, 0.349111700475108]], 'mu': [0.0, 0.0, 0.0, 0.0], 'Sigma': [[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 0.0, 1.0]]}
[[2.2037593228771333, 0.5132408468120786, -0.5000000000000003, -0.3259617546687723], [0.5132408468120786, 2.0622629049088506, 0.32596175466877225, -0.4999999999999998], [-0.5000000000000003, 0.32596175466877225, 4.135873298336195, -1.1284994932968848], [-0.3259617546687723, -0.4999999999999998, -1.1284994932968848, 3.3604209799237017]]


In [4]:
spring_5_masses, cov_ss = generate_spring_mass(num_masses=5, seed=20260414)
print(spring_5_masses)
print(cov_ss)

{'name': 'ou', 'A': [[-0.0, -0.0, -0.0, -0.0, -0.0, -1.0, -0.0, -0.0, -0.0, -0.0], [-0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -1.0, -0.0, -0.0, -0.0], [-0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -1.0, -0.0, -0.0], [-0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -1.0, -0.0], [-0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -1.0], [2.4162711902967646, -1.1888939242795724, -0.0, -0.0, -0.0, 0.517537700940358, -0.0, -0.0, -0.0, -0.0], [-0.913327164816163, 1.7523847978292248, -0.8390576330130619, -0.0, -0.0, -0.0, 0.3646300153338944, -0.0, -0.0, -0.0], [-0.0, -1.014802051571491, 1.9541610649918424, -0.9393590134203516, -0.0, -0.0, -0.0, 0.36251607517625134, -0.0, -0.0], [-0.0, -0.0, -0.8511173119489894, 1.8427811367979636, -0.9916638248489744, -0.0, -0.0, -0.0, 0.31618581179451966, -0.0], [-0.0, -0.0, -0.0, -0.8796956811933355, 1.6702782098141953, -0.0, -0.0, -0.0, -0.0, 0.3326747722235176]], 'mu': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], 'Sigma': [[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 

In [5]:
spring_10_masses, cov_ss = generate_spring_mass(num_masses=10, seed=20260414)
print(spring_10_masses)
print(cov_ss)

{'name': 'ou', 'A': [[-0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -1.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0], [-0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -1.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0], [-0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -1.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0], [-0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -1.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0], [-0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -1.0, -0.0, -0.0, -0.0, -0.0, -0.0], [-0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -1.0, -0.0, -0.0, -0.0, -0.0], [-0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -1.0, -0.0, -0.0, -0.0], [-0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -1.0, -0.0, -0.0], [-0